# Librerias utilizadas

Se usaron las siguientes librerias para la realización del proyecto

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Limpieza del dataset

Se empezó por la limpieza y agrupación de los datos presentes en el dataset utilizado en la realización del proyecto

In [ ]:
# Diccionario de las regiones del Censo con sus estados.
regiones = {'Northeast': ['Connecticut', 'Maine', 'Massachusetts', 'New Hampshire', 'Rhode Island', 'Vermont', 
                          'New Jersey', 'New York', 'Pennsylvania'],
            'Midwest': ['Illinois', 'Indiana', 'Michigan', 'Ohio', 'Wisconsin', 'Iowa', 'Kansas', 'Minnesota', 
                        'Missouri', 'Nebraska', 'North Dakota', 'South Dakota'],
            'South': ['Delaware', 'District Of Columbia', 'Florida', 'Georgia', 'Maryland', 'North Carolina',
                      'South Carolina', 'Virginia', 'West Virginia', 'Alabama', 'Kentucky','Mississippi', 
                      'Tennessee', 'Arkansas', 'Louisiana', 'Oklahoma', 'Texas'],
            'West':['Arizona', 'Colorado', 'Idaho', 'Montana', 'Nevada', 'New Mexico', 'Utah', 'Wyoming', 'Alaska',
                    'California', 'Hawaii', 'Oregon', 'Washington']
}

def asignar_region(estado):
    """
    Asigna una región geografica del Censo a los estados de Estados Unidos

    Args:
        estado (str): Nombre del estado a clasificar. 

    Returns:
        srt: Nombre de la región del censo a la que pertenece el estado o Sin Región si no pertenece a ninguna
        de las regiones.

    Example:
        >>> asignar_region('California')
        'West'
    """
    for region, estados in regiones.items():
        if estado in estados:
            return region
    return 'Sin Region'

def limpiar_agrupar():
    """
    Lee el dataset original, limpia los datos y los agrupa por región del Censo y por año.
    También elimina las filas con valores nulos y promedia las variables por años y región.

    Args:
        No recibe parametros

    Returns:
        DataFrame: Dataset limpio y agrupado por región y año.

    Example:
        >>> df_agrupado = limpiar_y_agrupar()
        Dataset limpio exportado a: ../dataset/dataset_limpio.csv
    """
    
    df = pd.read_csv('../dataset/aqi_yearly_1980_to_2021.csv')
    
    variables_a_utilizar = ['State', 'Year', 'Median AQI', 'Good Days', 'Moderate Days', 
                        'Unhealthy for Sensitive Groups Days', 'Unhealthy Days', 'Very Unhealthy Days', 
                        'Hazardous Days', 'Days CO', 'Days NO2', 'Days Ozone', 'Days SO2', 'Days PM2.5', 'Days PM10']
    
    # Elimina filas con valores nulos
    df = df.dropna(subset=variables_a_utilizar)

    # Crea la columna Region dentro de la tabla aplicando la función asignar_region a cada estado
    df['Region'] = df['State'].apply(asignar_region)

    # Elimina los registros que no pertenecen a ninguna región del census
    df = df[df['Region'] != 'Sin Region']

    # Columnas de tipos de días
    dias_cols = [
        'Good Days',
        'Moderate Days',
        'Unhealthy for Sensitive Groups Days',
        'Unhealthy Days',
        'Very Unhealthy Days',
        'Hazardous Days'
    ]

    # Total de días registrados por fila
    df['Total_Dias'] = df[dias_cols].sum(axis=1)

    # Porcentaje de cada tipo de día
    for col in dias_cols:
        df[col] = (df[col] / df['Total_Dias'])

    # Columnas de contaminantes
    contaminantes_cols = [
        'Days CO',
        'Days NO2',
        'Days Ozone',
        'Days SO2',
        'Days PM2.5',
        'Days PM10'
    ]

    # Total de días por contaminante
    df['Total_Contaminantes'] = df[contaminantes_cols].sum(axis=1)

    # Porcentaje de cada contaminante
    for col in contaminantes_cols:
        df[col] = (df[col] / df['Total_Contaminantes'])

    # Agrupar por región y año, y promediando las variables a utilizar
    df_agrupado = df.groupby(['Region','Year'])[['Median AQI','Good Days','Moderate Days', 
                                                 'Unhealthy for Sensitive Groups Days','Unhealthy Days', 
                                                 'Very Unhealthy Days','Hazardous Days','Days CO','Days NO2', 
                                                 'Days Ozone','Days SO2','Days PM2.5','Days PM10']].mean().reset_index()
    
    df_agrupado.to_csv('../dataset/dataset_limpio.csv', index = False)

    return df_agrupado

df_agrupado = limpiar_agrupar()

# Interpolación de datos

Se aplicó el método de splines cúbicos para analizar graficamente la tendencia del AQI en las distintas regiones de los Estados Unidos

In [ ]:
df = pd.read_csv("../dataset/dataset_limpio.csv")


# Spline cúbico Natural

def splineNatural(n: int, x: list, a: list):
    """
    Realiza la interpolación por splines naturales para un conjunto de puntos iniciales.
    Los datos de tipo lista deben ingresarse como listas en sus respectivas variables.
    
    Args:
        n (int): Número de intervalos.
        x (list): Lista de puntos en el eje x.
        a (list): Lista de valores de la función evaluada en cada punto x.
    
    Returns:
        list[tuple]: Lista de tuplas con los coeficientes de cada polinomio del spline.
    
    Example:
        >>> x = [0, 1, 2]
        >>> a = [-5, -4, 3]
        >>> splineNatural(2, x, a)
        [(-5, -0.5, 0.0, 1.5), (-4, 4.0, 4.5, -1.5)]
    """

    h = [0] * n

    i = 0
    while i < n:
        h[i] = x[i+1] - x[i]
        i += 1

    alpha = [0] * n

    j = 1
    while j < n:
        alpha[j] = ((3 / h[j]) * (a[j+1] - a[j])) - \
                   ((3 / h[j-1]) * (a[j] - a[j-1]))
        j += 1

    l = [0] * (n + 1)
    u = [0] * (n + 1)
    z = [0] * (n + 1)

    l[0] = 1
    u[0] = 0
    z[0] = 0

    j = 1
    while j < n:

        l[j] = 2 * (x[j+1] - x[j-1]) - h[j-1] * u[j-1]
        u[j] = h[j] / l[j]
        z[j] = (alpha[j] - h[j-1] * z[j-1]) / l[j]
        j += 1

    b = [0] * n
    c = [0] * (n + 1)
    d = [0] * n

    l[n] = 1
    z[n] = 0
    c[n] = 0

    j = n - 1

    while j >= 0:

        c[j] = z[j] - u[j] * c[j+1]
        b[j] = ((a[j+1] - a[j]) / h[j]) - (h[j] * (c[j+1] + 2 * c[j]) / 3)
        d[j] = (c[j+1] - c[j]) / (3 * h[j])
        j -= 1

    resultado = []

    j = 0
    while j < n:

        resultado.append((a[j], b[j], c[j], d[j]))
        j += 1

    return resultado


# Evaluación del spline

def evaluarSpline(x_eval: float, x: list, coeficientes: list):
    """
    Evalúa el spline cúbico natural en un punto específico
    Los datos de tipo lista se ingresan como una lista en la variable respectiva

    Args:
        x_eval (float): Punto en el que se desea evaluar el spline.
        x (list): Lista de puntos en el eje x.
        coeficientes (list[tuple]): Lista de tuplas con los coeficientes de cada polinomio del spline.

    Returns:
        float: Valor del spline evaluado en x_eval.

    Example:
        >>> x_eval = 1.5
        >>> x = [0, 1, 2]
        >>> coeficientes = [
        ...     (-5, -0.5, 0.0, 1.5),
        ...     (-4, 4.0, 4.5, -1.5)
        ... ]
        >>> evaluar_spline(x_eval, x, coeficientes)
        0.625
    """

    n = len(coeficientes)

    for i in range(n):
        if x[i] <= x_eval <= x[i+1]:
            a, b, c, d = coeficientes[i]
            dx = x_eval - x[i]
            return a + b * dx + c * (dx**2) + d * (dx**3)

    raise ValueError("El punto de evaluación está fuera del rango de los datos")    

def graficar_spline_por_region(df):
    """
    Grafica el spline cúbico de cada región en una subgráfica separada.

    Args:
        df (DataFrame): Dataset limpio y agrupado por región y año.

    Returns:
        None
    """
    for region in df["Region"].unique():
        datos = df[df["Region"] == region].sort_values("Year")
        x = datos["Year"].tolist()
        y = datos["Median AQI"].tolist()
        n = len(x) - 1

        coeficientes = splineNatural(n, x, y)
        x_suavizado = np.linspace(min(x), max(x), 500)
        y_suavizado = [evaluarSpline(xi, x, coeficientes) for xi in x_suavizado]

        plt.figure(figsize=(10, 5))
        plt.plot(x_suavizado, y_suavizado, color="blue", label="Spline")
        plt.scatter(x, y, color="red", zorder=5, label="Datos originales")
        plt.title(f"Spline Cúbico - Región {region}")
        plt.xlabel("Año")
        plt.ylabel("Median AQI")
        plt.legend()
        plt.grid(True)
        plt.tight_layout()
        plt.savefig(f"../graficas/spline_{region}.png")
        plt.show()  

def graficar_dias_por_region(df):
    """
    Grafica la distribución de días por categoría de AQI por región.

    Args:
        df (DataFrame): Dataset limpio y agrupado por región y año.

    Returns:
        None
    """
    dias = ['Good Days', 'Moderate Days', 'Unhealthy for Sensitive Groups Days',
                     'Unhealthy Days', 'Very Unhealthy Days', 'Hazardous Days']

    for region in df['Region'].unique():
        datos = df[df['Region'] == region].sort_values('Year')
        años = datos['Year'].tolist()

        plt.figure(figsize=(10, 5))
        for col in dias:
            plt.plot(años, datos[col].tolist(), label=col)

        plt.title(f'Distribución de días por categoría - Región {region}')
        plt.xlabel('Año')
        plt.ylabel('Porcentaje de días (%)')
        plt.legend(fontsize=6)
        plt.grid(True)
        plt.tight_layout()
        plt.savefig(f'../graficas/dias_{region}.png')
        plt.show()

def graficar_contaminantes_por_region(df):
    """
    Grafica el porcentaje de días dominados por cada contaminante por región.

    Args:
        df (DataFrame): Dataset limpio y agrupado por región y año.

    Returns:
        None
    """
    contaminantes = ['Days CO', 'Days NO2', 'Days Ozone','Days SO2', 'Days PM2.5', 'Days PM10']

    for region in df['Region'].unique():
        datos = df[df['Region'] == region].sort_values('Year')
        años = datos['Year'].tolist()

        plt.figure(figsize=(10, 5))
        for col in contaminantes:
            plt.plot(años, datos[col].tolist(), label=col)

        plt.title(f'Contaminantes dominantes - Región {region}')
        plt.xlabel('Año')
        plt.ylabel('Porcentaje de días (%)')
        plt.legend(fontsize=6)
        plt.grid(True)
        plt.tight_layout()
        plt.savefig(f'../graficas/contaminantes_{region}.png')
        plt.show()

def graficar_spline_combinado(df):
    """
    Grafica los datos originales y la curva del spline cúbico natural para todas las regiones

    Args:
        df (DataFrame): Dataset limpio y agrupado por región y año.

    Returns:
        None
    """
    plt.figure(figsize=(12, 6))

    for region in df["Region"].unique():
        datos = df[df["Region"] == region].sort_values("Year")
        x = datos["Year"].tolist()
        y = datos["Median AQI"].tolist()
        n = len(x) - 1

        coeficientes = splineNatural(n, x, y)

        x_suavizado = np.linspace(min(x), max(x), 500)
        y_suavizado = [evaluarSpline(xi, x, coeficientes) for xi in x_suavizado]

        plt.plot(x_suavizado, y_suavizado, label=f"Spline {region}")
        plt.scatter(x, y, zorder=5)

    plt.title("Tendencia del Median AQI por Región")
    plt.xlabel("Año")
    plt.ylabel("Median AQI")
    plt.legend()
    plt.grid(True)
    plt.savefig("../graficas/spline_combinado.png")
    plt.show()
    

graficar_spline_por_region(df)
graficar_dias_por_region(df)
graficar_contaminantes_por_region(df)
graficar_spline_combinado(df)

# Extrapolación de datos

Se aplicó minimos cuadrados sobre los datos para obtener las respectivas gráficas que permitieran aproximar la tendencia a futuro del AQI en las distintas regiones de los Estados Unidos

In [ ]:
df = pd.read_csv("../dataset/dataset_limpio.csv")

def minimos_cuadrados(x, y, grado):
    """
    Ajusta un polinomio de grado especificado mediante el método de mínimos cuadrados
    y devuelve los coeficientes y la ecuación ajustada.

    Args:
        x (list[float]): Años históricos.
        y (list[float]): Valores históricos del Median AQI.
        grado (int): Grado del polinomio a ajustar.

    Returns:
        tuple: Coeficientes del polinomio y ecuación ajustada en formato string.

    Example:
        >>> coef, ecuacion = minimos_cuadrados(x=[2000,2001,2002], y=[50,48,46], grado=2)
        'y = -0.026x^2 + 0.155x + 1.155'
    """
    x = np.array(x)
    y = np.array(y)
    A = np.vander(x, grado + 1)
    coef, *_ = np.linalg.lstsq(A, y, rcond=None)

    terminos = []

    for i, c in enumerate(coef):
        exp = grado - i
        if exp > 1:
            terminos.append(f"{c:.3f}x^{exp}")
        elif exp == 1:
            terminos.append(f"{c:.3f}x")
        else:
            terminos.append(f"{c:.3f}")
    
    ecuacion = "y = " + " + ".join(terminos)
    return coef, ecuacion

def mostrar_valores_prediccion(df, grado, año_fin):
    """
    Muestra una tabla con los valores predichos del Median AQI por región y año futuro.

    Args:
        df (DataFrame): Dataset limpio y agrupado por región y año.
        grado (int): Grado del polinomio a ajustar.
        año_fin (int): Año hasta el que se quiere predecir.

    Returns:
        DataFrame: Tabla con los valores predichos por región y año.

    Example:
        >>> mostrar_valores_prediccion(df, grado=1, año_fin=2030)
    """
    años_prediccion = np.arange(2022, año_fin + 1)
    resultados = {'Año': años_prediccion}

    for region in df['Region'].unique():
        datos = df[df['Region'] == region].sort_values('Year')
        x = datos['Year'].tolist()
        y = datos['Median AQI'].tolist()

        coef, _ = minimos_cuadrados(x, y, grado)
        aqi_predicho = np.polyval(coef, años_prediccion)

        resultados[region] = np.round(aqi_predicho, 2)

    df_prediccion = pd.DataFrame(resultados)
    print(df_prediccion.to_string(index=False))
    return df_prediccion

def graficar_prediccion_por_region(df, grado, año_fin):
    """
    Grafica la predicción de mínimos cuadrados de cada región en una subgráfica separada,
    mostrando los datos históricos, la curva ajustada, la predicción y la ecuación.

    Args:
        df (DataFrame): Dataset limpio y agrupado por región y año.
        grado (int): Grado del polinomio a ajustar.
        año_fin (int): Año hasta el que se quiere predecir.

    Returns:
        None

    Example:
        >>> graficar_prediccion_por_region(df, grado=2, año_fin=2030)
    """
    for region in df['Region'].unique():
        datos = df[df['Region'] == region].sort_values('Year')
        x = datos['Year'].tolist()
        y = datos['Median AQI'].tolist()

        coef, ecuacion = minimos_cuadrados(x, y, grado)

        años_prediccion = np.arange(x[0], año_fin + 1)
        aqi_predicho = np.polyval(coef, años_prediccion)

        plt.figure(figsize=(10, 5))
        plt.plot(x, y, 'o', color='red', zorder=5, label='Datos históricos')
        plt.plot(años_prediccion, aqi_predicho, '-', color='blue', label='Predicción')
        plt.axvline(x=x[-1], color='gray', linestyle='--', label='Inicio predicción')
        plt.title(f'Predicción Mínimos Cuadrados - Región {region}')
        plt.xlabel('Año')
        plt.ylabel('Median AQI')
        plt.text(0.05, 0.05, ecuacion, transform=plt.gca().transAxes, fontsize=7, verticalalignment='bottom')
        plt.legend(loc='upper right')
        plt.grid(True)
        plt.tight_layout()
        plt.savefig(f'../graficas/prediccion_{region}.png')
        plt.show()

def graficar_prediccion_combinado(df, grado, año_fin):
    """
    Grafica la predicción de mínimos cuadrados de todas las regiones en una misma figura.

    Args:
        df (DataFrame): Dataset limpio y agrupado por región y año.
        grado (int): Grado del polinomio a ajustar.
        año_fin (int): Año hasta el que se quiere predecir.

    Returns:
        None

    Example:
        >>> graficar_prediccion_combinado(df, grado=2, año_fin=2030)
    """
    plt.figure(figsize=(12, 6))

    for region in df['Region'].unique():
        datos = df[df['Region'] == region].sort_values('Year')
        x = datos['Year'].tolist()
        y = datos['Median AQI'].tolist()

        coef, ecuacion = minimos_cuadrados(x, y, grado)

        años_prediccion = np.arange(x[0], año_fin + 1)
        aqi_predicho = np.polyval(coef, años_prediccion)

        plt.plot(x, y, 'o', zorder=5)
        plt.plot(años_prediccion, aqi_predicho, '-', label=f'{region}: {ecuacion}')
        plt.axvline(x=x[-1], color='gray', linestyle='--')

    plt.title('Predicción del Median AQI por Región - Mínimos Cuadrados')
    plt.xlabel('Año')
    plt.ylabel('Median AQI')
    plt.legend(fontsize=7)
    plt.grid(True)
    plt.savefig('../graficas/prediccion_combinado.png')
    plt.show()

mostrar_valores_prediccion(df, grado=2, año_fin=2030)
graficar_prediccion_por_region(df, grado=2, año_fin=2030)
graficar_prediccion_combinado(df, grado=2, año_fin=2030)